# hMOF Dataset Splits

In [13]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Locate project root regardless of who runs this notebook
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / ".git").exists():
        ROOT = _p
        break
else:
    ROOT = Path.cwd()
os.chdir(ROOT)
print(f"Project root: {ROOT}")

PARQUET_PATH = ROOT / "data" / "cleaned" / "hmof_dataset_cleaned.parquet"
SPLITS_DIR   = ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

CO2_COLS = [
    "co2_mol_kg_0.01bar", "co2_mol_kg_0.05bar", "co2_mol_kg_0.1bar",
    "co2_mol_kg_0.5bar",  "co2_mol_kg_2.5bar",
]
RANDOM_STATE = 42

df = pd.read_parquet(PARQUET_PATH)

print(f"Loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"Topologies: {sorted(df['topology'].unique())}")

Project root: /Users/isabellamueller-vogt/Library/Mobile Documents/com~apple~CloudDocs/05 - ETH/06 - FS 2026/04 - Digital Chemistry/dc_project
Loaded: 111,469 rows × 27 columns
Topologies: ['acs', 'bct', 'bcu', 'cpf', 'cpr', 'dia', 'fcu', 'fnu', 'fsc', 'fse', 'fsf', 'fsg', 'hcb', 'hex', 'hms', 'hxl', 'ilc', 'irl', 'jeb', 'lfm', 'llj', 'mot', 'nbo', 'pcu', 'rna', 'rob', 'sit', 'sql', 'sqp', 'tbo']


In [14]:
def partition_stats(name: str, split: dict[str, pd.DataFrame]) -> dict:
    """Print per-partition stats and return a summary dict for the comparison table."""
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    row = {"split": name}
    for part, sub in split.items():
        n          = len(sub)
        n_topo     = sub["topology"].nunique()
        n_metal    = sub["metal_node"].nunique()
        co2_mean   = sub[CO2_COLS].mean().mean()
        co2_std    = sub[CO2_COLS].std().mean()
        print(
            f"  {part:6s}  n={n:>7,}  topologies={n_topo:>3}  "
            f"metal_nodes={n_metal:>3}  CO2_mean={co2_mean:.3f}  CO2_std={co2_std:.3f}"
        )
        row[f"{part}_n"]       = n
        row[f"{part}_topo"]    = n_topo
        row[f"{part}_metal"]   = n_metal
        row[f"{part}_co2mean"] = round(co2_mean, 3)
    return row


def assert_no_leakage(train: pd.DataFrame, test: pd.DataFrame, col: str) -> None:
    train_vals = set(train[col].dropna())
    assert set(test[col].dropna()).isdisjoint(train_vals), f"test/{col} leaks into train"
    print(f"zero {col} leakage between train and test")


Split 1: Random (90 / 10)

Hyperparameters are tuned via cross-validation on the training set.

In [15]:
def make_stratify_col(series: pd.Series, min_count: int = 3) -> pd.Series:
    """
    Return a stratify-safe copy of `series`: classes with fewer than
    `min_count` members are relabelled '__OTHER__' so sklearn can split them.
    """
    counts  = series.value_counts()
    rare    = counts[counts < min_count].index.tolist()
    strat   = series.copy()
    strat[strat.isin(rare)] = "__OTHER__"
    if rare:
        print(f"Merged {len(rare)} rare class(es) into '__OTHER__': {rare}")
    else:
        print("No rare classes, all classes have >= min_count samples.")
    return strat


print("Stratify key: topology")
strat_col = make_stratify_col(df["topology"])

train_rand, test_rand = train_test_split(
    df, test_size=0.10, random_state=RANDOM_STATE, stratify=strat_col
)

summary_rand = partition_stats(
    "Random 90/10",
    {"train": train_rand, "test": test_rand}
)

# Save
train_rand.to_parquet(SPLITS_DIR / "split_random_train.parquet", index=False)
test_rand .to_parquet(SPLITS_DIR / "split_random_test.parquet",  index=False)


Stratify key: topology
Merged 8 rare class(es) into '__OTHER__': ['lfm', 'fsf', 'acs', 'cpr', 'sit', 'ilc', 'irl', 'jeb']

Random 90/10
  train   n=100,322  topologies= 30  metal_nodes= 22  CO2_mean=1.573  CO2_std=1.094
  test    n= 11,147  topologies= 21  metal_nodes= 13  CO2_mean=1.565  CO2_std=1.100


Split 2: Topology-stratified (unseen topologies in test)

pcu is always pinned to train -> it makes up ~90% of all structures, so assigning it
randomly risks it landing in the test set and making the train set tiny.
The remaining 29 topologies are split randomly into train and test groups.

In [16]:
def group_split(df: pd.DataFrame, group_col: str,
                test_size: float = 0.10,
                random_state: int = RANDOM_STATE) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hold out entire groups (topologies or metal nodes) for the test set.
    Uses GroupShuffleSplit to assign whole groups to train or test.
    Returns (df_train, df_test).
    """
    groups = df[group_col].values
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, groups=groups))
    return df.iloc[train_idx].copy(), df.iloc[test_idx].copy()


# Pin pcu to train; split remaining topologies randomly into train / test
df_pcu     = df[df["topology"] == "pcu"].copy()
df_non_pcu = df[df["topology"] != "pcu"].copy()

print(f"pcu structures (always in train): {len(df_pcu):,}")
print(f"Non-pcu structures:               {len(df_non_pcu):,}")
print(f"Non-pcu topologies:               {df_non_pcu['topology'].nunique()}")

train_non_pcu, test_topo = group_split(df_non_pcu, group_col="topology")
train_topo = pd.concat([df_pcu, train_non_pcu], ignore_index=True)

assert_no_leakage(train_topo, test_topo, "topology")

summary_topo = partition_stats(
    "Topology-stratified",
    {"train": train_topo, "test": test_topo}
)

# Save
train_topo.to_parquet(SPLITS_DIR / "split_topology_train.parquet", index=False)
test_topo .to_parquet(SPLITS_DIR / "split_topology_test.parquet",  index=False)


pcu structures (always in train): 100,462
Non-pcu structures:               11,007
Non-pcu topologies:               29
zero topology leakage between train and test

Topology-stratified
  train   n=111,409  topologies= 27  metal_nodes= 22  CO2_mean=1.572  CO2_std=1.095
  test    n=     60  topologies=  3  metal_nodes=  8  CO2_mean=0.824  CO2_std=0.770


Split 3: Metal-node-stratified (unseen metal nodes in test)

In [17]:
df_with_metal = df[df["metal_node"].notna()].copy()
df_no_metal   = df[df["metal_node"].isna()].copy()

train_metal_base, test_metal = group_split(df_with_metal, group_col="metal_node")

# Structures with no metal-node label go into train (they cannot be used for leakage testing)
train_metal = pd.concat([train_metal_base, df_no_metal], ignore_index=True)

assert_no_leakage(train_metal, test_metal, "metal_node")

summary_metal = partition_stats(
    "Metal-node-stratified",
    {"train": train_metal, "test": test_metal}
)

# Save
train_metal.to_parquet(SPLITS_DIR / "split_metal_train.parquet", index=False)
test_metal .to_parquet(SPLITS_DIR / "split_metal_test.parquet",  index=False)


zero metal_node leakage between train and test

Metal-node-stratified
  train   n= 82,235  topologies= 28  metal_nodes= 20  CO2_mean=1.605  CO2_std=1.080
  test    n= 29,234  topologies= 15  metal_nodes=  3  CO2_mean=1.479  CO2_std=1.128


## Summary comparison

In [18]:
summary = pd.DataFrame([summary_rand, summary_topo, summary_metal]).set_index("split")

# Reorder columns for readability
parts = ["train", "test"]
cols  = [f"{p}_{s}" for p in parts for s in ["n", "topo", "metal", "co2mean"]]
summary = summary[cols]

summary


,train_n,train_topo,train_metal,train_co2mean,test_n,test_topo,test_metal,test_co2mean
split,,,,,,,,
Random 90/10,100322,30,22,1.573,11147,21,13,1.565
Topology-stratified,111409,27,22,1.572,60,3,8,0.824
Metal-node-stratified,82235,28,20,1.605,29234,15,3,1.479
